# Prompt Evaluation in AWS Bedrock


This project demonstrates how to evaluate LLM outputs using Amazon Bedrock. It focuses on structured tasks such as generating **Python, JSON, and Regex**.

## Approach

We use three evaluation methods:

* **Code graders** → Validate syntax (fast, rule-based)
* **Model graders** → Use an LLM to score and review outputs
* **Human graders** → Manual evaluation (conceptual)

## Goal

Build a simple and scalable pipeline to assess the quality of model-generated responses.



## 📦 Importing Libraries

* **boto3**: AWS SDK for Python, used to interact with Amazon Bedrock and invoke foundation models.
* **json**: Built-in Python library for parsing and handling JSON data, used for datasets and evaluation results.


In [ ]:
import boto3
import json
import re
import ast
from statistics import mean

## 💬 Bedrock Client & Chat Helper

In this section, we set up the **Amazon Bedrock client** and define helper functions to interact with the model.

* Initialize the Bedrock runtime client using **boto3**
* Specify the **Claude 3.5 Haiku model** for fast evaluations
* Create helper functions to structure messages (user & assistant roles)
* Define a `chat` function to send requests to the model and return responses

This simplifies how we communicate with the LLM throughout the notebook.


In [42]:
client = boto3.client("bedrock-runtime", region_name="us-west-2")
# Use Haiku for faster evals
model_id = "us.anthropic.claude-3-5-haiku-20241022-v1:0"


def add_user_message(messages, text):
    user_message = {"role": "user", "content": [{"text": text}]}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": [{"text": text}]}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "modelId": model_id,
        "messages": messages,
        "inferenceConfig": {
            "temperature": temperature,
            "stopSequences": stop_sequences,
        },
    }

    if system:
        params["system"] = [{"text": system}]

    response = client.converse(**params)

    return response["output"]["message"]["content"][0]["text"]

## 🧪 Dataset Generation

In this section, we use the LLM to automatically **generate evaluation tasks**.

* A prompt is defined to create tasks related to **Python, JSON, and Regex for AWS**
* The model returns structured data in **JSON format**
* Each task includes a description, expected format, and evaluation criteria
* The response is parsed into a Python object using `json.loads`

This allows us to quickly create test cases for evaluating prompt performance.

In [60]:
def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
        "format": "json" or "python" or "regex",
        "solution_criteria": "Key criteria for evaluating the solution"
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [61]:
dataset = generate_dataset()
with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

## 🚀 Running Prompts

In this section, we **send tasks to the model** to generate solutions.

* Combines each evaluation task with a **system prompt** describing the model role
* Requests the model to respond **only with Python, JSON, or Regex**, without explanations
* Uses the `chat` helper to send the prompt and receive the model’s output
* Returns the generated solution for further evaluation

This step produces the model responses that we will later grade.

In [ ]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}

* Respond only with Python, JSON, or a plain Regex
* Do not add any comments or commentary or explanation
"""

    system_prompt = "You are an experienced AWS engineer with a hyper focus on addressing corner cases"

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```code")
    output = chat(messages, stop_sequences=["```"], system=system_prompt)
    return output

## 📝 Model-Based Grading

This section uses the LLM to **evaluate generated solutions** automatically.

* Sends the task, model output, and evaluation criteria to the model
* Instructs the model to return a **structured JSON evaluation** with:

  * `strengths` – key positives
  * `weaknesses` – areas for improvement
  * `reasoning` – concise assessment
  * `score` – numeric rating (1–10)
* Parses the model’s response with `json.loads` for further analysis

This allows semantic and qualitative evaluation beyond simple syntax checks.


In [62]:
def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Criteria you should use to evaluate the solution:
<criteria>
{test_case["solution_criteria"]}
</criteria>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    """

    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)


## ✅ Code-Based (Syntax) Grading

This section implements **automatic syntax checks** for model outputs:

* **JSON** → Parsed with `json.loads`
* **Python** → Checked with `ast.parse`
* **Regex** → Compiled with `re.compile`

The `grade_syntax` function selects the appropriate validator based on the task format and returns a score:

* `10` → Valid syntax
* `0` → Invalid syntax

This method provides a fast, deterministic way to assess basic correctness before semantic evaluation.


In [1]:
def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0


def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0


def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0


def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)


## 🧩 Running & Grading a Test Case

This section **combines prompt execution and grading**:

* Calls `run_prompt` to generate the model’s solution for a task
* Uses `grade_by_model` for semantic evaluation (strengths, weaknesses, reasoning, score)
* Uses `grade_syntax` for deterministic syntax validation
* Computes the **average score** from model-based and syntax-based grading
* Returns a structured result including the output, reasoning, and final score

This provides an end-to-end evaluation for each test case.


In [63]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    model_grade = grade_by_model(test_case, output)
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    syntax_score = grade_syntax(output, test_case)

    score = (model_score + syntax_score) / 2

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning,
    }

## 📊 Running Evaluation on a Dataset

This section **runs all test cases in the dataset** and aggregates results:

* Iterates through each test case and calls `run_test_case`
* Collects outputs, scores, and reasoning for each task
* Computes the **average score** across all tasks
* Returns a list of structured results for analysis

This provides a full evaluation pipeline for the dataset, combining syntax and model-based grading.

In [ ]:

def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")

    return results

In [65]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

Average score: 8.833333333333334



* The **average score** is `8.83` out of `10`
* This indicates that, on average, the model’s outputs are **high quality**
* **Strengths**:

  * Most solutions are syntactically correct
  * The model generally meets the solution criteria
* **Potential weaknesses**:

  * Minor improvements could be made in handling edge cases or formatting
  * Some outputs may need slight semantic refinement

**Conclusion:** The model performs very well on these tasks, but reviewing individual cases can help identify **specific areas for improvement**.


In [ ]:
print(json.dumps(results, indent=2))

[
  {
    "output": "\n^[a-z0-9][a-z0-9-]{1,61}[a-z0-9]$\n",
    "test_case": {
      "task": "Create a regular expression to validate an AWS S3 bucket name (lowercase letters, numbers, hyphens, 3-63 characters long)",
      "format": "regex",
      "solution_criteria": "Matches valid S3 bucket naming conventions, prevents special characters, enforces length constraints"
    },
    "score": 9.0,
    "reasoning": "The regex captures most S3 bucket naming rules with good precision. It handles length, character set, and start/end character constraints effectively. However, it misses a few nuanced AWS S3 bucket naming edge cases that would require additional validation."
  },
  {
    "output": "\nimport re\n\ndef parse_arn(arn):\n    pattern = r'^arn:aws:(?P<service>[^:]+):(?P<region>[^:]*):(?P<account_id>[^:]+):(?P<resource_type>[^:/]+)(?:/|:)(?P<resource_id>.+)$'\n    match = re.match(pattern, arn)\n    \n    if not match:\n        return None\n    \n    return {\n        'service': matc

From your dataset, the **average score is ~8.83/10**, and looking at individual results:

1. **S3 Bucket Regex** → 9.0

   * Strong handling of naming rules (length, allowed characters, start/end constraints)
   * Minor edge cases not fully covered

2. **ARN Parser (Python)** → 8.5

   * Correctly extracts most ARN components
   * Could improve handling of all ARN variations and deeper validation

3. **Lambda Function JSON** → 9.0

   * Complete CloudFormation structure with key properties
   * Covers basics well; could be extended for full deployment scenarios

**Overall:**

* The model produces **high-quality outputs** with good syntax and mostly correct logic.
* Minor improvements are possible in **edge-case handling and coverage**.
* Both code-based syntax checks and model-based grading confirm that the solutions are **robust and reliable for AWS tasks**.
